In [ ]:
import os
import time
import gc
import torch
import pandas as pd
import numpy as np
import random
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict, ClassLabel
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

# ---------------------------------------------------------
# [환경 설정]
# ---------------------------------------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
gc.collect()
torch.cuda.empty_cache()

SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

## 데이터셋 로드

In [ ]:
# ---------------------------------------------------------
# 1. 데이터 로드 및 전처리 (Data Loading & Preprocessing)
# ---------------------------------------------------------
print("\n🔄 데이터 로드 및 병합 작업 시작...")

# (1) 기존 데이터 로드 (Old)
PATH_OLD = "./data/munch_labeled_FINAL_190139.csv"
df_old = pd.read_csv(PATH_OLD)

# Old 데이터 전처리
# 필수 컬럼에 결측치가 있으면 제거
df_old = df_old.dropna(subset=['input_sentence', 'intensity', 'field', 'difficulty_label'])
# Label을 정수형으로 변환
df_old['label'] = df_old['difficulty_label'].astype(int)

# (2) 신규 데이터 로드 (New - 이미 메타데이터 작업 완료됨)
PATH_NEW = "./data/munch_labeled_RESUMED_200125_revised.csv"
df_new = pd.read_csv(PATH_NEW)

# New 데이터 전처리
# 필수 컬럼 결측치 제거
df_new = df_new.dropna(subset=['input_sentence', 'intensity', 'field', 'difficulty_label'])
# Label을 정수형으로 변환
df_new['label'] = df_new['difficulty_label'].astype(int)

# ---------------------------------------------------------
# 2. Feature Engineering (포맷 통일)
# ---------------------------------------------------------
# 두 데이터셋의 포맷(띄어쓰기, 대소문자 등)을 완벽히 일치시키기 위해
# df_new에 'text'가 있더라도 다시 생성합니다.

def combine_features(row):
    # 안전하게 문자열로 변환 후 대문자 처리
    intensity = str(row['intensity']).upper().strip()
    field = str(row['field']).upper().strip()
    sentence = str(row['input_sentence']).strip()
    
    # 포맷: [INTENSITY] [FIELD] 문장
    return f"[{intensity}] [{field}] {sentence}"

# 양쪽 데이터프레임에 동일 함수 적용
df_old['text'] = df_old.apply(combine_features, axis=1)
df_new['text'] = df_new.apply(combine_features, axis=1)


# ---------------------------------------------------------
# 3. 병합 (Merge)
# ---------------------------------------------------------
# 필요한 컬럼만 뽑아서 위아래로 합침
cols = ['text', 'label']
df_final = pd.concat([df_old[cols], df_new[cols]], ignore_index=True)

print(f"📊 데이터 병합 완료!")
print(f"   - 기존 데이터(Old): {len(df_old)}개")
print(f"   - 신규 데이터(New): {len(df_new)}개")
print(f"   - 최종 학습 데이터 : {len(df_final)}개")
print(f"   - 데이터 예시: {df_final.iloc[0]['text']}")

In [ ]:
# ---------------------------------------------------------
# 4. 데이터셋 생성 및 모델 준비
# ---------------------------------------------------------
dataset = Dataset.from_pandas(df_final)

# ClassLabel 캐스팅 (Stratify 필수)
dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=["Easy", "Hard"]))

# 데이터 분할 (Train/Test)
train_test_split = dataset.train_test_split(test_size=0.15, seed=SEED, stratify_by_column="label")
dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

# 모델 & 토크나이저
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 클래스 가중치 계산 (Weighted Loss용)
train_labels = dataset_dict['train']['label']
class_weights = compute_class_weight("balanced", classes=np.unique(train_labels), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️ Class Weights: Easy(0)={class_weights[0]:.2f}, Hard(1)={class_weights[1]:.2f}")

# 토크나이징
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 모델 로드

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# ---------------------------------------------------------
# 1. 학습된 V3 모델 로드
# ---------------------------------------------------------
MODEL_PATH = "./model/smart_router_v3_final_merged" 
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## 스마트 라우터 결과 문장을 저장하기

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import os
from tqdm import tqdm

def save_full_test_results(model, tokenizer, test_dataset, threshold=0.25, save_path="./output/full_test_results.csv"):
    """
    테스트 데이터셋 전체에 대해 추론을 수행하고, 상세 결과를 CSV로 저장합니다.
    """
    print(f"\n🚀 [Smart Router V3] 전체 테스트셋 추론 시작 (Threshold: {threshold})")
    print(f"📌 데이터셋 크기: {len(test_dataset)}건")
    print("=" * 80)

    model.eval()
    device = next(model.parameters()).device
    
    results_list = []
    
    # 진행률 표시를 위해 tqdm 사용
    for i in tqdm(range(len(test_dataset)), desc="Processing"):
        sample = test_dataset[i]
        
        # 1. 입력 데이터 준비
        inputs = {
            'input_ids': torch.tensor([sample['input_ids']]).to(device),
            'attention_mask': torch.tensor([sample['attention_mask']]).to(device)
        }
        
        # 2. 정답(Label) 확인
        true_label = sample['label'] # 0(Easy) or 1(Hard)
        
        # 3. 원본 텍스트 복원
        # 스페셜 토큰([CLS], [SEP]) 제거
        full_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
        
        # 4. 모델 추론
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)
            hard_prob = probs[0][1].item() # Class 1 (Hard) 확률

        # 5. 라우팅 결정 (Threshold 적용)
        pred_label = 1 if hard_prob >= threshold else 0
        difficulty_str = "Hard" if pred_label == 1 else "Easy"
        
        # 6. 채점 (정답 여부)
        is_correct = (pred_label == true_label)
        
        # 7. 라우팅 타겟 모델
        target_model = "gpt-4" if pred_label == 1 else "gemini-flash"

        # 8. 태그 파싱 (추가 기능)
        # 텍스트에서 [TAG] 부분 추출 시도 (예: "[STRONG] [NONE] 문장...")
        try:
            parts = full_text.split('] ', 2) # '] ' 기준으로 앞에서 2개만 자름
            if len(parts) >= 3:
                intensity_tag = parts[0].replace('[', '').strip()
                field_tag = parts[1].replace('[', '').strip()
                pure_sentence = parts[2]
            else:
                # 태그 형식이 안 맞을 경우 예외 처리
                intensity_tag = "UNKNOWN"
                field_tag = "UNKNOWN"
                pure_sentence = full_text
        except:
            intensity_tag = "ERROR"
            field_tag = "ERROR"
            pure_sentence = full_text

        # 9. 결과 리스트에 추가
        results_list.append({
            "index": i,
            "full_input_text": full_text,       # 전체 텍스트 (태그 포함)
            "input_sentence": pure_sentence,    # 순수 문장
            "tag_intensity": intensity_tag,     # 추출한 강도 태그
            "tag_field": field_tag,             # 추출한 분야 태그
            "true_label": true_label,           # 실제 정답 (0/1)
            "pred_label": pred_label,           # 예측값 (0/1)
            "hard_prob": round(hard_prob, 4),   # Hard 확률 (소수점 4자리)
            "threshold": threshold,             # 적용된 임계값
            "difficulty": difficulty_str,       # Easy / Hard
            "routed_model": target_model,       # 배정된 모델
            "is_correct": is_correct            # 정답 여부 (True/False)
        })

    # DataFrame 변환
    df_results = pd.DataFrame(results_list)
    
    # 최종 정확도 계산
    accuracy = df_results['is_correct'].mean() * 100
    
    print("\n" + "=" * 80)
    print(f"🏆 [최종 분석 리포트]")
    print(f"   - 전체 데이터 수 : {len(df_results)}건")
    print(f"   - 전체 정확도    : {accuracy:.2f}%")
    print(f"   - 정답 건수      : {df_results['is_correct'].sum()}건")
    print(f"   - 오답 건수      : {(~df_results['is_correct']).sum()}건")
    print("=" * 80)

    # 저장
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df_results.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"💾 결과 파일 저장 완료: {save_path}")
    
    return df_results

# ---------------------------------------------------------
# 실행 (전체 데이터셋)
# ---------------------------------------------------------
# 저장 경로 설정
OUTPUT_CSV_PATH = "./output/router_full_test_results.csv"
SELECTED_THRESHOLD = 0.25 

# 함수 실행
df_final = save_full_test_results(
    model=model, 
    tokenizer=tokenizer, 
    test_dataset=tokenized_datasets["test"], 
    threshold=SELECTED_THRESHOLD, 
    save_path=OUTPUT_CSV_PATH
)

# 데이터 미리보기
print("\n[저장된 데이터 샘플]")
display(df_final.head())

## 스마트 라우터 결과 문장을 실제 모델에 입력하기

### 분류 결과 분포 확인하기

In [ ]:
df1 = pd.read_csv("./output/router_full_test_results.csv")

In [ ]:
pd.crosstab(df1['is_correct'], df1['routed_model'])

확인 결과 해당 서비스 기간 고성능 모델로는 gpt 4.1, 경량 모델로는 gemini 2.0 flash를 사용했음을 확인함

## 실행 결과(smart routing -> llm 모델 입력)와 원본 정답 문장(sentencify.ai)과 비교하여 품질 측정하기

In [ ]:
import time
from google import genai
from google.genai import types
from tqdm import tqdm

# =========================================================
# [Mock API Functions] 
# =========================================================
def call_gemini_flash(text, intensity, field):
    # 실제로는 google.generativeai 라이브러리 사용
    # prompt = f"Correct this text: '{text}' with {intensity} intensity for {field} field."
    time.sleep(0.5) # API latency simulation
    return f"[Flash Correction] ({intensity}/{field}) {text} -> (Corrected by Flash)"

def call_gpt4(text, intensity, field):
    # 실제로는 openai 라이브러리 사용
    time.sleep(0.8) 
    return f"[GPT-4 Correction] ({intensity}/{field}) {text} -> (Corrected by GPT-4)"

In [ ]:
# =========================================================
# [Execution Loop] 라우팅 결과에 따라 모델 호출
# =========================================================
generated_results = []

print("🚀 [End-to-End Evaluation] 실제 모델 교정 시작...")

for index, row in tqdm(df1.iterrows(), total=len(df1)):
    # 입력 데이터 추출 (파싱된 태그 활용)
    input_text = row['input_sentence']
    intensity = row['tag_intensity']
    field = row['tag_field']
    target_model = row['routed_model'] # 'gemini-flash' or 'gpt-4'
    
    # 라우팅된 모델에 따라 API 호출
    try:
        if target_model == 'gemini-flash':
            output_text = call_gemini_flash(input_text, intensity, field)
        else:
            output_text = call_gpt4(input_text, intensity, field)
    except Exception as e:
        output_text = f"Error: {str(e)}"

    # 결과 저장
    generated_results.append(output_text)

# 데이터프레임에 결과 추가
df1['final_output'] = generated_results

In [ ]:
# =========================================================
# [저장] 최종 결과 (채점 전)
# =========================================================
df1.to_csv("./output/2.final_generation_results.csv", index=False, encoding='utf-8-sig')
print("💾 생성 결과 저장 완료: ./output/2.final_generation_results.csv")

# 결과 미리보기
display(df1[['input_sentence', 'routed_model', 'final_output']].head())